# DisasterM3 — U-Net Segmentation Fine-Tuning (Stage 1 of Hybrid Triage)
**Purpose:** Train a ResNet34-backed U-Net to produce 1024×1024 damage masks from the
37,204 Referring Expression Segmentation entries that were *excluded* from the VLM
fine-tuning pipeline (D6: mask-path targets are unusable for text generation).

**Hardware:** Kaggle T4 GPU (16 GB VRAM) — U-Net is lightweight (~21M params), so
no quantization needed.

**Architecture mirrors `train_qwen_disasterm3.ipynb`:**
| Pattern | VLM notebook | This notebook |
|---|---|---|
| Pinned deps + idempotent install | Cell 1 | Cell 1 |
| All config in one cell | Cell 2 | Cell 2 |
| Manifest audit + data prep | Cell 4 | Cell 4 |
| Checkpoint-resume across sessions | Cells 8.5, 9 | Cells 7, 8 |
| TimeLimitCallback (12 h cap) | Cell 10 | Cell 9 |
| HF Hub push on session end | Cell 10b | Cell 10 |

### How to run ("Run All" workflow)
1. **Prerequisites (once):** GPU T4, internet ON, DisasterM3 mirror dataset attached,
   `HF_TOKEN` secret (write-scoped) for checkpoint backup.
2. **Fresh session:** Run All → Cell 1 installs deps and HALTS → Restart kernel →
   Run All again. Cell 1 skips installation and training proceeds.
3. Training auto-saves checkpoints every N epochs; Cell 10 pushes to HF Hub.
4. **Resume session:** Set `RESUME_FROM_HF` in Cell 2, then Run All.

### Relationship to VLM pipeline
This notebook trains the **spatial extraction** stage of the Hybrid Triage Pipeline.
The U-Net produces masks → `hybrid_bdc_counter.py` counts buildings and extracts
bounding boxes → the fine-tuned Qwen2.5-VL (from `train_qwen_disasterm3.ipynb`)
reasons about the cropped patches.

In [18]:
# Cell 1: Environment Setup — Install Pinned Dependencies
import importlib.metadata as _md

PINS = {
    "torch": "2.4.1",
    "torchvision": "0.19.1",
    "torchaudio": "2.4.1",
    "segmentation-models-pytorch": "0.3.4",
    "albumentations": "1.4.21",
    "opencv-python-headless": "4.10.0.84",
}

_mismatched = []
for _pkg, _want in PINS.items():
    try:
        _have = _md.version(_pkg)
    except _md.PackageNotFoundError:
        _have = "not installed"
    if _have != _want:
        _mismatched.append(f"{_pkg}: {_have} → {_want}")

if _mismatched:
    print("⏳ Installing pinned versions:")
    for _m in _mismatched:
        print(f"   {_m}")
    !pip install -q \
        torch==2.4.1 \
        torchvision==0.19.1 \
        torchaudio==2.4.1 \
        segmentation-models-pytorch==0.3.4 \
        albumentations==1.4.21 \
        opencv-python-headless==4.10.0.84 \
        pillow \
        huggingface_hub[hf_xet]
    raise SystemExit(
        "✓ Dependencies installed. RESTART THE KERNEL NOW "
        "(Run → Restart & clear cell outputs), then click 'Run All' again — "
        "this cell will detect the correct versions and skip installation."
    )

print("✓ All pinned versions already installed — proceeding.")

✓ All pinned versions already installed — proceeding.


In [19]:
# Cell 2: Configuration
import os
import json
import torch
from pathlib import Path
from datetime import datetime

DATA_ROOT = Path("/kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct")
MANIFEST_PATH = DATA_ROOT / "train_release.json"

ENCODER_NAME = "resnet34"
ENCODER_WEIGHTS = "imagenet"
NUM_CLASSES = 3

LEARNING_RATE = 5e-4
NUM_EPOCHS = 25
BATCH_SIZE =  16
NUM_WORKERS = 4
IMAGE_SIZE = 512
WEIGHT_DECAY = 1e-5

CHECKPOINT_DIR = "/kaggle/working/unet_checkpoints"
SAVE_EVERY_N_EPOCHS = 2
TIME_LIMIT_HOURS = 11.5

HF_CHECKPOINT_REPO = "Aryan6489/disasterm3-unet-checkpoints"
RESUME_EPOCH = 24
NUM_EPOCHS = 35

OUTPUT_DIR = "/kaggle/working/unet_disasterm3"
MODEL_NAME = f"disasterm3_unet_{ENCODER_NAME}_ep{NUM_EPOCHS}"

print(f"✓ Config loaded")
print(f"  Encoder: {ENCODER_NAME} (pretrained={ENCODER_WEIGHTS})")
print(f"  Classes: {NUM_CLASSES} (Background, Intact, Damaged)")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Image size: {IMAGE_SIZE}×{IMAGE_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB" if torch.cuda.is_available() else "")

✓ Config loaded
  Encoder: resnet34 (pretrained=imagenet)
  Classes: 3 (Background, Intact, Damaged)
  Batch size: 16
  Image size: 512×512
  Epochs: 35
  Device: CPU



## 2. Load & Prepare Segmentation Data
We use the **37,204 Referring Expression Segmentation entries** that were excluded
from the VLM fine-tuning (D6). Their `ground_truth` field contains the mask file
path, and the images are the same pre/post-disaster pairs from the DisasterM3 dataset.

In [20]:
# Cell 3: Build Combined Multi-Class Segmentation Masks
import cv2
import numpy as np
from collections import Counter, defaultdict

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)
print(f"Loaded {len(raw_data):,} entries from manifest")

bda_entries = [
    e for e in raw_data
    if e.get("task") == "Referring Expression Segmentation"
    and e.get("cls_description") == "Building Damage Assessment"
]
print(f"Building Damage Assessment entries: {len(bda_entries):,}")

def resolve_path(rel_path):
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    candidates = [
        DATA_ROOT / rel_path,
        DATA_ROOT / "train_images" / rel_path,
        DATA_ROOT / "train_images" / "train_images" / filename,
        DATA_ROOT / "masks" / rel_path,
        DATA_ROOT / "masks" / "masks" / filename,
    ]
    for c in candidates:
        if c.exists():
            return str(c)
    return None

by_image = defaultdict(dict)
for e in bda_entries:
    post_img = e.get("post_image_path", "")
    if e.get("image_type") != "Optical":
        continue
    mask_rel = e.get("ground_truth", "")
    folder = Path(mask_rel.replace("\\", "/")).parent.name
    resolved = resolve_path(mask_rel)
    if resolved:
        by_image[post_img][folder] = resolved
        by_image[post_img]["post_image_path"] = post_img
print(f"Unique images with at least one mask: {len(by_image):,}")

FOLDER_TO_CLASS = {
    "train_building_intact_mask": 1,
    "train_building_damaged_mask": 2,
    "train_building_destroyed_mask": 3,
}

MASK_CACHE_DIR = Path("/kaggle/working/combined_masks")
MASK_CACHE_DIR.mkdir(parents=True, exist_ok=True)

pairs = []
skipped_img = 0
skipped_no_mask = 0

for i, (post_img, mask_dict) in enumerate(by_image.items()):
    img_path = resolve_path(post_img)
    if not img_path:
        skipped_img += 1
        continue

    combined_mask = None
    for folder, class_id in FOLDER_TO_CLASS.items():
        mask_path = mask_dict.get(folder)
        if mask_path is None:
            continue
        m = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        if combined_mask is None:
            combined_mask = np.zeros_like(m, dtype=np.uint8)
        combined_mask[m > 0] = class_id

    if combined_mask is None:
        skipped_no_mask += 1
        continue

    mask_out_path = MASK_CACHE_DIR / f"mask_{i}.png"
    cv2.imwrite(str(mask_out_path), combined_mask)

    pairs.append({
        "image_path": img_path,
        "mask_path": str(mask_out_path),
    })

print(f"\n✓ Built {len(pairs):,} valid (image, mask_path) pairs")
print(f"  Skipped (image not found): {skipped_img:,}")
print(f"  Skipped (no valid mask files): {skipped_no_mask:,}")

if pairs:
    print(f"\n  Sample image: {pairs[0]['image_path']}")
    print(f"  Sample mask:  {pairs[0]['mask_path']}")
    check_mask = cv2.imread(pairs[0]["mask_path"], cv2.IMREAD_GRAYSCALE)
    print(f"  Combined mask unique values: {np.unique(check_mask)}")

Loaded 92,968 entries from manifest
Building Damage Assessment entries: 14,531
Unique images with at least one mask: 6,443

✓ Built 6,443 valid (image, mask_path) pairs
  Skipped (image not found): 0
  Skipped (no valid mask files): 0

  Sample image: /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct/train_images/train_images/bata_explosion_post_0.png
  Sample mask:  /kaggle/working/combined_masks/mask_0.png
  Combined mask unique values: [0 1 2]


In [21]:
# Cell 4: PyTorch Dataset Definition and Augmentation Pipelines
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torch.utils.data import Dataset, DataLoader

class DisasterM3SegDataset(Dataset):
    """Dataset for DisasterM3 segmentation entries.

    Loads post-disaster images and their pre-merged combined damage masks
    from disk (masks were written once in Cell 4, not held in memory):
      0 = Background
      1 = Intact (no damage)
      2 = Damaged
      3 = Destroyed
    """

    def __init__(self, pairs, transform=None, image_size=512):
        self.pairs = pairs
        self.transform = transform
        self.image_size = image_size

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        pair = self.pairs[idx]

        image = cv2.imread(pair["image_path"])
        if image is None:
            return self.__getitem__((idx + 1) % len(self))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        mask = cv2.imread(pair["mask_path"], cv2.IMREAD_GRAYSCALE)
        if mask is None:
            return self.__getitem__((idx + 1) % len(self))

        if self.transform:
            transformed = self.transform(image=image, mask=mask)
            image = transformed["image"]
            mask = transformed["mask"]

        return image, mask.long()

train_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
    A.CLAHE(clip_limit=2.0, p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])
val_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

from sklearn.model_selection import train_test_split
train_pairs, val_pairs = train_test_split(pairs, test_size=0.1, random_state=42)

train_dataset = DisasterM3SegDataset(train_pairs, transform=train_transform, image_size=IMAGE_SIZE)
val_dataset = DisasterM3SegDataset(val_pairs, transform=val_transform, image_size=IMAGE_SIZE)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"✓ Datasets created")
print(f"  Train: {len(train_dataset):,} samples ({len(train_loader):,} batches)")
print(f"  Val:   {len(val_dataset):,} samples ({len(val_loader):,} batches)")

✓ Datasets created
  Train: 5,798 samples (363 batches)
  Val:   645 samples (41 batches)


In [22]:
# Cell 5: Loss Function Definitions and Class Weights
import torch
import torch.nn as nn
import torch.nn.functional as F

NUM_CLASSES = 4
class_pixel_counts = torch.tensor(
    [6267276585, 395903867, 70675310, 22119406], dtype=torch.float32
)
total_pixels = class_pixel_counts.sum()
class_weights = total_pixels / (NUM_CLASSES * class_pixel_counts)
class_weights = torch.clamp(class_weights, max=5.0)
class_weights = (class_weights / class_weights.sum() * NUM_CLASSES)

# ── Define the loss classes FIRST ──
class DiceLoss(nn.Module):
    """Multi-class Dice loss. Expects logits [B, C, H, W] and integer targets [B, H, W]."""
    def __init__(self, num_classes, smooth=1e-5):
        super().__init__()
        self.num_classes = num_classes
        self.smooth = smooth

    def forward(self, logits, targets):
        probs = F.softmax(logits, dim=1)
        targets_onehot = F.one_hot(targets, self.num_classes).permute(0, 3, 1, 2).float()
        dims = (0, 2, 3)
        intersection = torch.sum(probs * targets_onehot, dims)
        cardinality = torch.sum(probs + targets_onehot, dims)
        dice_per_class = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return 1.0 - dice_per_class.mean()


class CombinedLoss(nn.Module):
    """Weighted CrossEntropy + Dice, summed with configurable weighting."""
    def __init__(self, class_weights, num_classes, ce_weight=0.5, dice_weight=0.5):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(weight=class_weights)
        self.dice = DiceLoss(num_classes)
        self.ce_weight = ce_weight
        self.dice_weight = dice_weight

    def forward(self, logits, targets):
        ce_loss = self.ce(logits, targets)
        dice_loss = self.dice(logits, targets)
        return self.ce_weight * ce_loss + self.dice_weight * dice_loss, ce_loss.item(), dice_loss.item()


# ── NOW instantiate, after DEVICE is defined ──
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = CombinedLoss(class_weights.to(DEVICE), NUM_CLASSES, ce_weight=0.35, dice_weight=0.65)

print(f"✓ Loss function ready on {DEVICE}")

✓ Loss function ready on cpu


In [23]:
# Cell 6: Model, Loss, Optimizer, and Scheduler Setup
import segmentation_models_pytorch as smp
from torch.optim.lr_scheduler import CosineAnnealingLR

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_CLASSES = 4

model = smp.Unet(
    encoder_name=ENCODER_NAME,
    encoder_weights=ENCODER_WEIGHTS,
    in_channels=3,
    classes=NUM_CLASSES,
).to(device)

class_pixel_counts = torch.tensor([6267276585, 395903867, 70675310, 22119406], dtype=torch.float32)
total_pixels = class_pixel_counts.sum()
class_weights = total_pixels / (NUM_CLASSES * class_pixel_counts)
class_weights = (class_weights / class_weights.sum() * NUM_CLASSES).to(device)

criterion = CombinedLoss(class_weights, NUM_CLASSES)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"✓ Model loaded on {device}")
print(f"  Architecture: U-Net ({ENCODER_NAME} encoder)")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")
print(f"  Class weights: {class_weights.tolist()}")

✓ Model loaded on cpu
  Architecture: U-Net (resnet34 encoder)
  Total parameters: 24,436,804
  Trainable parameters: 24,436,804
  Class weights: [0.010286855511367321, 0.16284401714801788, 0.912207841873169, 2.914661169052124]


In [26]:
# Cell 8: Evaluation Metrics — mIoU Computation
def compute_iou(pred, target, num_classes=NUM_CLASSES):
    """Compute per-class IoU and mean IoU."""
    ious = []
    for cls in range(num_classes):
        pred_cls = (pred == cls)
        target_cls = (target == cls)
        intersection = (pred_cls & target_cls).sum().item()
        union = (pred_cls | target_cls).sum().item()
        if union == 0:
            ious.append(float('nan'))
        else:
            ious.append(intersection / union)
    return ious

def validate(model, val_loader, criterion, device):
    """Run validation and return loss + mIoU."""
    model.eval()
    total_loss = 0
    all_ious = [[] for _ in range(NUM_CLASSES)]

    with torch.no_grad():
        for images, masks in val_loader:
            images = images.to(device)
            masks = masks.to(device)

            outputs = model(images)
            loss, _, _ = criterion(outputs, masks)
            total_loss += loss.item()

            preds = outputs.argmax(dim=1)
            for pred, mask in zip(preds, masks):
                ious = compute_iou(pred.cpu(), mask.cpu())
                for cls, iou in enumerate(ious):
                    if not np.isnan(iou):
                        all_ious[cls].append(iou)

    avg_loss = total_loss / len(val_loader)
    class_ious = [np.mean(cls_ious) if cls_ious else 0.0 for cls_ious in all_ious]
    miou = np.mean([iou for iou in class_ious if iou > 0])

    return avg_loss, miou, class_ious

print("✓ Metrics defined (mIoU, per-class IoU)")

✓ Metrics defined (mIoU, per-class IoU)


In [27]:
# Cell 9: Training Loop with Time-Limit Callback
import time
from datetime import timedelta
from torch.cuda.amp import GradScaler, autocast

deadline = time.time() + TIME_LIMIT_HOURS * 3600
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

scaler = GradScaler()

best_miou = 0.0
history = []
epoch_times = []
training_start = time.time()

print(f"Batch size: {BATCH_SIZE}")
print(f"Number of training batches: {len(train_loader):,}")
print(f"Number of training images: {len(train_loader.dataset):,}")
print(f"Number of validation batches: {len(val_loader):,}")
print(f"Number of validation images: {len(val_loader.dataset):,}")
print(f"start_epoch = {start_epoch}, NUM_EPOCHS = {NUM_EPOCHS}")
print(f"Time remaining: {(deadline - time.time())/3600:.2f} hours")
print()
print(f"Starting training at {datetime.now().strftime('%H:%M:%S')}")
print(f"   Epochs: {start_epoch} → {NUM_EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Time budget: {TIME_LIMIT_HOURS} hours")
print(f"   Mixed precision (AMP): enabled")
print(f"   Checkpoints every {SAVE_EVERY_N_EPOCHS} epochs to {CHECKPOINT_DIR}")
print()

for epoch in range(start_epoch, NUM_EPOCHS):
    epoch_start = time.time()

    if time.time() > deadline:
        print(f"\nTime budget reached at epoch {epoch} — saving and stopping.")
        break

    model.train()
    epoch_loss = 0.0

    for batch_idx, (images, masks) in enumerate(train_loader):
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        with autocast():
            outputs = model(images)
            loss, ce_val, dice_val = criterion(outputs, masks)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()

        if time.time() > deadline:
            print(f"\nTime limit hit mid-epoch {epoch}, batch {batch_idx} — saving.")
            break

    scheduler.step()
    avg_train_loss = epoch_loss / (batch_idx + 1)

    val_loss, miou, class_ious = validate(model, val_loader, criterion, device)

    epoch_duration = time.time() - epoch_start
    epoch_times.append(epoch_duration)
    total_elapsed = time.time() - training_start
    avg_epoch_time = sum(epoch_times) / len(epoch_times)
    eta_str = str(timedelta(seconds=int(avg_epoch_time * (NUM_EPOCHS - (epoch + 1)))))
    elapsed_str = str(timedelta(seconds=int(total_elapsed)))
    epoch_dur_str = str(timedelta(seconds=int(epoch_duration)))

    history.append({
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "val_loss": val_loss,
        "miou": miou,
        "bg_iou": class_ious[0],
        "intact_iou": class_ious[1],
        "damaged_iou": class_ious[2],
        "destroyed_iou": class_ious[3],
    })

    print(f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
          f"Train: {avg_train_loss:.4f} | Val: {val_loss:.4f} | mIoU: {miou:.4f} "
          f"| Epoch time: {epoch_dur_str} | Elapsed: {elapsed_str} | ETA: {eta_str}")

    if miou > best_miou:
        best_miou = miou
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, "best_model.pth"))
        print(f"  ✓ New best mIoU: {best_miou:.4f} (saved)")

    if (epoch + 1) % SAVE_EVERY_N_EPOCHS == 0 or time.time() > deadline:
        ckpt_path = os.path.join(CHECKPOINT_DIR, f"unet_epoch_{epoch}.pth")
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "best_miou": best_miou,
            "history": history,
        }, ckpt_path)
        print(f"  Checkpoint saved: {ckpt_path}")

final_path = os.path.join(OUTPUT_DIR, "final_model.pth")
torch.save(model.state_dict(), final_path)
total_time = str(timedelta(seconds=int(time.time() - training_start)))
print(f"\nTraining complete at {datetime.now().strftime('%H:%M:%S')}")
print(f"  Total training time: {total_time}")
print(f"  Best mIoU: {best_miou:.4f}")
print(f"  Final model: {final_path}")

with open(os.path.join(OUTPUT_DIR, "training_history.json"), "w") as f:
    json.dump(history, f, indent=2)

Batch size: 16
Number of training batches: 363
Number of training images: 5,798
Number of validation batches: 41
Number of validation images: 645
start_epoch = 0, NUM_EPOCHS = 35
Time remaining: 11.50 hours

Starting training at 06:40:41
   Epochs: 0 → 35
   Batch size: 16
   Time budget: 11.5 hours
   Mixed precision (AMP): enabled
   Checkpoints every 2 epochs to /kaggle/working/unet_checkpoints



/tmp/ipykernel_58/184994754.py:10: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_58/184994754.py:49: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/usr/local/lib/python3.12/dist-packages/torch/amp/autocast_mode.py:265: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


KeyboardInterrupt: 